In [45]:
import math
import os
import sys
import getpass
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

import string
import random


In [46]:
ds = load_dataset("cimec/lambada")

print(len(ds["train"]["text"]))
print(len(ds["validation"]["text"]))


2662
4869


In [47]:
class CharTokenizer:
    def __init__(self, text: str):
        self.special = ["<pad>", "<bos>", "<eos>", "<unk>"]
        chars = sorted(set(text)) + self.special

        self.stoi = {c: i for i, c in enumerate(chars)}
        self.itos = {i: c for c, i in self.stoi.items()}
        self.vocab_size = len(chars)

        self.pad_id = self.stoi["<pad>"]
        self.eos_id = self.stoi["<eos>"]
        self.unk_id = self.stoi["<unk>"]

    def encode(self, text: str, add_eos=True):
        ids = [self.stoi.get(c, self.unk_id) for c in text]
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)

In [48]:
def pad(seq: torch.Tensor, target_len: int, pad_id):
    seq = seq[:target_len]
    if len(seq) < target_len:
        seq = torch.cat([
            seq,
            torch.full((target_len - len(seq),), pad_id, dtype=torch.long)
        ])
    return seq

In [49]:
def build_packed_stream(tokenized_texts, eos_id):
    lengths = sum(len(seq) + 1 for seq in tokenized_texts)

    arr = torch.empty(lengths, dtype=torch.long)

    i = 0
    for seq in tokenized_texts:
        n = len(seq)
        arr[i:i+n] = torch.tensor(seq, dtype=torch.long)
        i += n

        arr[i] = eos_id
        i += 1

    return arr

In [50]:
class PackedTextDataset(Dataset):
    def __init__(self, stream, block_size):
        self.stream = stream
        self.block_size = block_size

        self.n_blocks = (len(stream) - 1) // block_size

    def __len__(self):
        return self.n_blocks

    def __getitem__(self, idx):
        start = idx * self.block_size
        end = start + self.block_size + 1

        chunk = self.stream[start:end]

        x = chunk[:-1]
        y = chunk[1:]

        return x, y

In [51]:
class WordEmbedding(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, context_len: int):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_len, embed_dim)
        self.context_len = context_len

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        tok = self.token_emb(idx)

        pos = torch.arange(T, device=idx.device)
        pos = self.pos_emb(pos)[None, :, :]

        return tok + pos

In [52]:
class MaskedMultiHeadSelfAttn(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, context_len: int):
        super().__init__()
        
        self.embed_dim   = embed_dim
        self.num_heads   = num_heads

        self.head_dim    = embed_dim // num_heads

        self.q_proj    = nn.Linear(embed_dim, embed_dim)
        self.k_proj    = nn.Linear(embed_dim, embed_dim)
        self.v_proj    = nn.Linear(embed_dim, embed_dim)
        self.out_proj    = nn.Linear(embed_dim, embed_dim) # final linear layer to process all concatenated attentions

        mask = torch.tril(torch.ones(context_len, context_len))

        self.register_buffer("mask", mask.view(1, 1, context_len, context_len))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_scores = attn_scores.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))

        attn_weights = torch.softmax(attn_scores, dim=-1)

        out = attn_weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(out)

In [53]:
class AttentionBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, context_len: int):
        super().__init__()

        self.ln1  = nn.LayerNorm(embed_dim)
        self.attn = MaskedMultiHeadSelfAttn(
            embed_dim=embed_dim,
            num_heads=num_heads,
            context_len=context_len
        )

        self.ln2  = nn.LayerNorm(embed_dim)

        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [54]:
CONTEXT_LEN  = 1024     # sequence length (tokens)
EMBED_DIM    = 256      # model width
NUM_HEADS    = 4        # attention heads  (must divide EMBED_DIM)
NUM_LAYERS   = 4        # transformer blocks
FF_DIM       = 512      # feed-forward hidden size

BATCH_SIZE   = 8
EPOCHS       = 1000
LR           = 3e-4
EVAL_SPLIT   = 0.1      # fraction of tokens held out for validation
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

DATA_PATH    = "data/textData.txt"

class TransformerSimple(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = EMBED_DIM, num_heads: int = NUM_HEADS, num_layers: int = NUM_LAYERS, ff_dim: int = FF_DIM, context_len: int = CONTEXT_LEN):
        super().__init__()

        self.embedding = WordEmbedding(vocab_size, embed_dim, context_len)
        
        self.blocks    = nn.Sequential(*[
            AttentionBlock(embed_dim, num_heads, ff_dim, context_len)
            for _ in range(num_layers)
        ])
        
        self.ln_f      = nn.LayerNorm(embed_dim)
        self.head      = nn.Linear(embed_dim, vocab_size, bias=False)

        self.head.weight = self.embedding.token_emb.weight

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor=None):
        x = self.embedding(idx)
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.head(x)

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            logits_flat = logits.reshape(B * T, V)
            targets_flat = targets.reshape(B * T)

            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, eos_id, max_new_tokens: int=128, temperature: float = 1.0, top_k: int=None) -> torch.Tensor:
        ctx = self.embedding.context_len
        B = idx.size(0)

        finished = torch.zeros(B, dtype=torch.bool, device=idx.device)

        for _ in range(max_new_tokens):

            if finished.all():
                break

            idx_cond = idx[:, -ctx:]
            logits, _ = self(idx_cond)

            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                thresh = v[:, -1].unsqueeze(-1)
                logits = torch.where(
                    logits < thresh,
                    torch.tensor(float("-inf"), device=logits.device),
                    logits
                )

            probs = torch.softmax(logits, dim=-1)
            next_t = torch.multinomial(probs, num_samples=1)

            idx = torch.cat([idx, next_t], dim=1)

            finished |= (next_t.squeeze(-1) == eos_id)

        return idx

In [55]:
def train(model: TransformerSimple, loader: DataLoader,
          optimizer: torch.optim.Optimizer  ) -> float:
    model.train()
    total_loss = 0.0
    i = 1
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        if i%(len(loader)//1000) == 0:
            return total_loss / i
            print(f"Batch Loss: {loss.item():.4f}, {round(i/len(loader), 2):.2f}")
        i += 1
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model: TransformerSimple, loader: DataLoader) -> float:
    model.eval()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        _, loss = model(x, y)

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def evaluate_accuracy(model: TransformerSimple, loader: DataLoader):
    model.eval()

    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        logits, _ = model(x)

        preds = logits.argmax(dim=-1)

        correct += (preds == y).sum().item()
        total += y.numel()

    return correct / total

In [56]:
data_set = load_dataset("cimec/lambada")

train_data_set = data_set["train"]

val_data_set = data_set["validation"]

stable_vocab = string.printable 

tokenizer = CharTokenizer(stable_vocab)
print(f"Vocabulary size : {tokenizer.vocab_size}")

train_tok = [torch.tensor(tokenizer.encode(item["text"]), dtype=torch.long) for item in train_data_set]
val_tok = [torch.tensor(tokenizer.encode(item["text"]), dtype=torch.long) for item in val_data_set]

print("Encoded")

train_stream = build_packed_stream(train_tok, tokenizer.eos_id)
print("Training converted to stream")
val_stream = build_packed_stream(val_tok, tokenizer.eos_id)
print("Validation converted to stream")

train_set  = PackedTextDataset(train_stream, CONTEXT_LEN)
val_set    = PackedTextDataset(val_stream, CONTEXT_LEN)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)

print(f"Train samples: {len(train_set):,}  |  Val samples: {len(val_set):,}")

Vocabulary size : 104
Encoded


C:\Users\prmcc\AppData\Local\Temp\ipykernel_19972\241627335.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  arr[i:i+n] = torch.tensor(seq, dtype=torch.long)


Training converted to stream
Validation converted to stream
Train samples: 955,155  |  Val samples: 1,634


In [57]:
transformerTiny = TransformerSimple(
    vocab_size  = tokenizer.vocab_size,
    embed_dim   = 128,
    num_heads   = 2,
    num_layers  = 2,
    ff_dim      = 256,
    context_len = CONTEXT_LEN,
).to(DEVICE)

transformerSmall = TransformerSimple(
    vocab_size  = tokenizer.vocab_size,
    embed_dim   = 256,
    num_heads   = 4,
    num_layers  = 6,
    ff_dim      = 1024,
    context_len = CONTEXT_LEN,
).to(DEVICE)

transformerMedium = TransformerSimple(
    vocab_size  = tokenizer.vocab_size,
    embed_dim   = 512,
    num_heads   = 8,
    num_layers  = 8,
    ff_dim      = 2048,
    context_len = CONTEXT_LEN,
).to(DEVICE)

transformerLarge = TransformerSimple(
    vocab_size  = tokenizer.vocab_size,
    embed_dim   = 768,
    num_heads   = 12,
    num_layers  = 12,
    ff_dim      = 3072,
    context_len = CONTEXT_LEN,
).to(DEVICE)

transformerMassive = TransformerSimple(
    vocab_size  = tokenizer.vocab_size,
    embed_dim   = 1024,
    num_heads   = 16,
    num_layers  = 16,
    ff_dim      = 4096,
    context_len = CONTEXT_LEN,
).to(DEVICE)

transformerList = [
    #{"name": "transformerTiny", "model": transformerTiny, "batch_size": 128, "optimizer": None, "scheduler": None}, 
    {"name": "transformerSmall", "model": transformerSmall, "batch_size": 128, "optimizer": None, "scheduler": None},
    #{"name": "transformerMedium", "model": transformerMedium, "batch_size": 64, "optimizer": None, "scheduler": None},
    #{"name": "transformerLarge", "model": transformerLarge, "batch_size": 16, "optimizer": None, "scheduler": None},
    #{"name": "transformerMassive", "model": transformerMassive, "batch_size": 8, "optimizer": None, "scheduler": None},
]

for i in transformerList:
    model = i["model"]
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{i["name"]} parameters : {n_params:,}\n")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
    i["optimizer"] = optimizer
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=LR / 10
    )
    i["scheduler"] = scheduler

transformerSmall parameters : 5,027,840



In [ ]:
#optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
#scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#    optimizer, T_max=EPOCHS, eta_min=LR / 10
#)

for epoch in range(1, EPOCHS + 1):
    for item in transformerList:
        print("Training model: " + item["name"])

        model = item["model"]
        optimizer = item["optimizer"]
        scheduler = item["scheduler"]

        train_loss = train(model, train_loader, optimizer)
        val_loss = evaluate(model, val_loader)
        val_acc = evaluate_accuracy(model, val_loader)

        scheduler.step()

        print(
            f"Epoch {epoch:>3}/{EPOCHS}  "
            f"train={train_loss:.4f}  "
            f"val={val_loss:.4f}  "
            f"ppl={math.exp(val_loss):.1f}  "
            f"acc={val_acc:.4%}"
        )

Training model: transformerSmall
Epoch   1/1000  train=2.5338  val=2.3869  ppl=10.9  acc=28.8029%
Training model: transformerSmall
Epoch   2/1000  train=2.3040  val=2.3779  ppl=10.8  acc=28.7177%
Training model: transformerSmall
Epoch   3/1000  train=2.2569  val=2.3990  ppl=11.0  acc=28.8541%
Training model: transformerSmall
Epoch   4/1000  train=2.2206  val=2.3737  ppl=10.7  acc=29.4786%
Training model: transformerSmall
Epoch   5/1000  train=2.1782  val=2.3324  ppl=10.3  acc=31.4270%
Training model: transformerSmall
Epoch   6/1000  train=2.0905  val=2.2214  ppl=9.2  acc=35.6816%
Training model: transformerSmall
Epoch   7/1000  train=1.9290  val=2.0366  ppl=7.7  acc=40.9060%
Training model: transformerSmall
Epoch   8/1000  train=1.7692  val=1.8636  ppl=6.4  acc=45.5266%
Training model: transformerSmall
Epoch   9/1000  train=1.6677  val=1.8219  ppl=6.2  acc=47.5585%
Training model: transformerSmall
Epoch  10/1000  train=1.5886  val=1.7652  ppl=5.8  acc=49.4486%
Training model: transform

In [ ]:
print("\n── Sample generation (temperature=0.8, top_k=40) ──\n")
model = None
for i in transformerList:
    if i["name"] == "transformerSmall":
        model = i["model"]
        break
    
model.eval()

seed_text = "the"
seed_ids  = torch.tensor(tokenizer.encode(seed_text, add_eos=False),
                            dtype=torch.long, device=DEVICE).unsqueeze(0)
out_ids   = model.generate(seed_ids, tokenizer.eos_id, 30,
                            temperature=1.0, top_k=40)
print(tokenizer.decode(out_ids[0].tolist()))


── Sample generation (temperature=0.8, top_k=40) ──

the ske , adectasthomon l ad whof
